In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE, ADASYN
from ctgan import CTGAN
import os

# Load the dataset
path = os.path.join("dataset.csv")
dataset = pd.read_csv(path)

# Drop unnecessary columns
dataset = dataset.drop(['seqn', 'Marital'], axis='columns')

# Map categorical variables to numerical values
sex_mapping = {'Male': 0, 'Female': 1}
race_mapping = {'White': 0, 'Asian': 1, 'Black': 2, 'MexAmerican': 3, 'Hispanic': 4, 'Other': 5}
dataset['Sex'] = dataset['Sex'].replace(sex_mapping)
dataset['Race'] = dataset['Race'].replace(race_mapping)

# Fill NaN values with the mean of the respective columns
dataset.iloc[:, 2] = dataset.iloc[:, 2].fillna(dataset.iloc[:, 2].mean())
dataset.iloc[:, 4] = dataset.iloc[:, 4].fillna(dataset.iloc[:, 4].mean())
dataset.iloc[:, 5] = dataset.iloc[:, 5].fillna(dataset.iloc[:, 5].mean())

# Clean the dataset before splitting
dataset = dataset.dropna(subset=['MetabolicSyndrome'])

# Split the data into training and test sets
outcome_0 = dataset[dataset['MetabolicSyndrome'] == 0]
outcome_1 = dataset[dataset['MetabolicSyndrome'] == 1]
test_size_each_class = 400
test_0 = outcome_0.sample(n=test_size_each_class, random_state=42)
test_1 = outcome_1.sample(n=test_size_each_class, random_state=42)
test_data = pd.concat([test_0, test_1])
train_data = dataset.drop(test_data.index)

# Define a function to generate synthetic samples using SMOTE, CTGAN, and ADASYN
def generate_synthetic_samples(train_data, weights=(0.33, 0.33, 0.34)):
    train_data = train_data.dropna(subset=['MetabolicSyndrome'])

    X = train_data.drop('MetabolicSyndrome', axis=1)
    y = train_data['MetabolicSyndrome']

    # Identify discrete (categorical) columns in your dataset
    discrete_columns = X.select_dtypes(include=['object', 'category']).columns.tolist()
    
    # Generate synthetic samples using SMOTE
    smote = SMOTE(random_state=42)
    X_smote, y_smote = smote.fit_resample(X, y)
    smote_samples = pd.DataFrame(X_smote[len(X):], columns=X.columns)

    # Generate synthetic samples using ADASYN
    adasyn = ADASYN(random_state=42)
    X_adasyn, y_adasyn = adasyn.fit_resample(X, y)
    adasyn_samples = pd.DataFrame(X_adasyn[len(X):], columns=X.columns)

    # Generate synthetic samples using CTGAN
    ctgan = CTGAN(epochs=100)
    ctgan.fit(X, discrete_columns)
    ctgan_samples = ctgan.sample(len(adasyn_samples))
    ctgan_samples.columns = X.columns  # Ensure columns match original data

    # Combine synthetic samples based on weights
    n_smote = int(weights[0] * len(smote_samples))
    n_ctgan = int(weights[1] * len(ctgan_samples))
    n_adasyn = max(0, len(smote_samples) - n_smote - n_ctgan)

    final_synthetic = pd.concat([
        smote_samples.sample(n=n_smote, random_state=42, replace=True),
        ctgan_samples.sample(n=n_ctgan, random_state=42, replace=True),
        adasyn_samples.sample(n=n_adasyn, random_state=42, replace=True)
    ])

    final_synthetic['MetabolicSyndrome'] = 1  # Assuming synthetic samples are for the minority class
    balanced_data = pd.concat([train_data, final_synthetic])

    return balanced_data

# Function to train and evaluate model
def evaluate_model(train_data, test_data):
    X_train = train_data.drop('MetabolicSyndrome', axis=1).values
    y_train = train_data['MetabolicSyndrome'].values
    X_test = test_data.drop('MetabolicSyndrome', axis=1).values
    y_test = test_data['MetabolicSyndrome'].values
    
    model = XGBClassifier(random_state=42, n_estimators=100, learning_rate=0.3, max_depth=3)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    return {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred)
    }

# Define a function to evaluate multiple weight combinations and store results in a CSV file
def evaluate_and_store_results(train_data, test_data):
    results_list = []

    # Iterate over lambda values for three methods
    for lambda_smote in np.arange(0, 1.05, 0.05):
        for lambda_ctgan in np.arange(0, 1.05 - lambda_smote, 0.05):
            lambda_adasyn = 1 - lambda_smote - lambda_ctgan
            weights = (lambda_smote, lambda_ctgan, lambda_adasyn)
            balanced_data_combined = generate_synthetic_samples(train_data, weights=weights)

            # Ensure sample size does not exceed population
            if len(balanced_data_combined) > len(train_data):
                balanced_data_combined = balanced_data_combined.sample(n=len(train_data), random_state=42, replace=True)

            results = evaluate_model(balanced_data_combined, test_data)

            # Store the results along with the lambda values and weights
            results_list.append({
                'lambda_smote': lambda_smote,
                'lambda_ctgan': lambda_ctgan,
                'lambda_adasyn': lambda_adasyn,
                'weights': weights,
                'accuracy': results['accuracy'],
                'precision': results['precision'],
                'recall': results['recall'],
                'f1': results['f1']
            })

    # Convert the results list to a DataFrame and save it as a CSV file
    results_df = pd.DataFrame(results_list)
    results_df.to_csv('smote_ctgan_adasyn_evaluation_results.csv', index=False)

# Evaluate and store the results in a CSV file
evaluate_and_store_results(train_data, test_data)
